# 4_evidently — Análise de Drift e Qualidade com Evidently AI

Objetivos principais:
- Carregar o dataset limpo e a pipeline treinada (`pipeline_log.joblib`);
- Gerar as predições para todo o dataset (com e sem correção de viés);
- Dividir em Referência (2006-2009) e Atual (2010);
- Gerar relatórios HTML do Evidently: Data Drift, Target Drift e Desempenho do Modelo;
- Executar Test Suites automatizadas com aprovação/reprovação (PASS/FAIL) e salvar relatórios em `runs/`.

In [ ]:
# Imports
from pathlib import Path
import json
import numpy as np
import pandas as pd
import joblib

# Tenta importar Evidently
try:
    try:
        from evidently import Report
        from evidently.presets import DataDriftPreset, RegressionPreset
        from evidently import TestSuite
        from evidently.tests import TestNumberOfDriftedColumns, TestValueMAE
    except ImportError:
        from evidently.legacy.report import Report
        from evidently.legacy.metric_preset import DataDriftPreset, RegressionPreset
        from evidently.legacy.test_suite import TestSuite
        from evidently.legacy.tests import TestNumberOfDriftedColumns, TestValueMAE

    EVIDENTLY_AVAILABLE = True
except Exception as e:
    EVIDENTLY_AVAILABLE = False
    print('Erro ao carregar Evidently:', e)

print('Evidently disponível:', EVIDENTLY_AVAILABLE)

Evidently disponível: True


In [6]:
# 1 — Carregar dados, pipeline e métricas
DATA_PATH = Path('..') / 'data' / 'processed' / 'ames_clean.parquet'
MODEL_PATH = Path('..') / 'models' / 'pipeline_log.joblib'
METRICS_PATH = Path('..') / 'models' / 'metrics.json'
RUNS_DIR = Path('..') / 'runs'
RUNS_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(DATA_PATH)
pipe = joblib.load(MODEL_PATH)
metrics = json.loads(METRICS_PATH.read_text()) if METRICS_PATH.exists() else {}
sigma_resid = metrics.get('sigma_resid_log', 0.0577)

print('Dados carregados:', df.shape)

Dados carregados: (2925, 80)


In [7]:
# 2 — Gerar predições e preparar conjuntos Ref / Cur
# Trata tipos de texto
for c in df.select_dtypes(include=['object']).columns:
    df[c] = df[c].astype('string')

X = df.drop(columns=['saleprice'])
y_log = np.log1p(df['saleprice'].astype(float))

# Predições em log e em dólares com correção de viés
pred_log = pipe.predict(X)
df['target'] = df['saleprice'].astype(float)
df['prediction'] = np.expm1(pred_log + 0.5 * sigma_resid**2)

# Divisão temporal: Ref (2006-2009) e Cur (2010)
ref_data = df[df['yr_sold'] < 2010].copy()
cur_data = df[df['yr_sold'] == 2010].copy()

print(f'Ref (2006-2009): {ref_data.shape[0]} linhas | Cur (2010): {cur_data.shape[0]} linhas')

Ref (2006-2009): 2584 linhas | Cur (2010): 341 linhas


/tmp/ipykernel_1503666/2038686109.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for c in df.select_dtypes(include=['object']).columns:


In [ ]:
# 3 — Gerar e salvar relatórios HTML com Evidently AI
if EVIDENTLY_AVAILABLE:
    # Monta a lista de presets suportados pela sua versão do Evidently
    metrics_list = [
        DataDriftPreset(),
        RegressionPreset()
    ]
    
    # Adiciona TargetDriftPreset apenas se tiver sido importado com sucesso
    if 'TargetDriftPreset' in globals() and TargetDriftPreset is not None:
        metrics_list.append(TargetDriftPreset())
    
    # Relatório Visual de Data Drift e Desempenho de Regressão
    report = Report(metrics=metrics_list)
    report.run(reference_data=ref_data, current_data=cur_data)
    report_path = RUNS_DIR / 'evidently_report.html'
    report.save_html(str(report_path))
    print('Relatório de Drift salvo em:', report_path)
    
    # Test Suite Automatizada (PASS / FAIL)
    tests = TestSuite(tests=[
        TestNumberOfDriftedColumns(lt=5),  # Menos de 5 colunas com drift
        TestValueMAE(gt=0)                 # Valida integridade do cálculo do erro
    ])
    tests.run(reference_data=ref_data, current_data=cur_data)
    test_path = RUNS_DIR / 'evidently_tests.html'
    tests.save_html(str(test_path))
    print('Test Suite salva em:', test_path)
else:
    print('Instale o evidently para gerar os arquivos HTML em runs/')

/l/disk0/arenno/Área de trabalho/INSPIRE3/EvidentlyIA/venv/lib/python3.12/site-packages/sklearn/metrics/_regression.py:1295: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)
/l/disk0/arenno/Área de trabalho/INSPIRE3/EvidentlyIA/venv/lib/python3.12/site-packages/sklearn/metrics/_regression.py:1295: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)
/l/disk0/arenno/Área de trabalho/INSPIRE3/EvidentlyIA/venv/lib/python3.12/site-packages/sklearn/metrics/_regression.py:1295: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)


Relatório de Drift salvo em: ../runs/evidently_report.html


/l/disk0/arenno/Área de trabalho/INSPIRE3/EvidentlyIA/venv/lib/python3.12/site-packages/sklearn/metrics/_regression.py:1295: UndefinedMetricWarning:

R^2 score is not well-defined with less than two samples.

/l/disk0/arenno/Área de trabalho/INSPIRE3/EvidentlyIA/venv/lib/python3.12/site-packages/sklearn/metrics/_regression.py:1295: UndefinedMetricWarning:

R^2 score is not well-defined with less than two samples.

/l/disk0/arenno/Área de trabalho/INSPIRE3/EvidentlyIA/venv/lib/python3.12/site-packages/sklearn/metrics/_regression.py:1295: UndefinedMetricWarning:

R^2 score is not well-defined with less than two samples.



Test Suite salva em: ../runs/evidently_tests.html


/l/disk0/arenno/Área de trabalho/INSPIRE3/EvidentlyIA/venv/lib/python3.12/site-packages/sklearn/metrics/_regression.py:1295: UndefinedMetricWarning:

R^2 score is not well-defined with less than two samples.

/l/disk0/arenno/Área de trabalho/INSPIRE3/EvidentlyIA/venv/lib/python3.12/site-packages/sklearn/metrics/_regression.py:1295: UndefinedMetricWarning:

R^2 score is not well-defined with less than two samples.

/l/disk0/arenno/Área de trabalho/INSPIRE3/EvidentlyIA/venv/lib/python3.12/site-packages/sklearn/metrics/_regression.py:1295: UndefinedMetricWarning:

R^2 score is not well-defined with less than two samples.

